# Re-baselining at `commitment=0`

`commitment-sweep.ipynb` and `commitment-confirmation.ipynb` established that
the old default of `commitment=2` was costing every policy that re-plans
meaningfully, and harming none that don't. `run_trial` now defaults to `0`.

That invalidates the headline table in `window-insertion-comparison.ipynb`,
which was measured at `commitment=2`. This notebook re-runs the same policy set
at both settings so the new baseline arrives with its provenance attached
rather than as a bare replacement.

**Why both settings rather than just the new one.** Every column here shares
`base_seed`, so trial *i* faces the same demand sequence at both commitment
levels. That makes the comparison paired, and a paired difference resolves a
two-point effect that independent confidence intervals barely separate. Running
only `commitment=0` would give the new numbers but no defensible statement
about what changed.

## What to expect

From the confirmation run, at 200 requests:

- `Fixed route` — exactly unchanged. A locked prefix of a fixed cycle is
  precisely what the policy would have returned anyway, so the two runs are
  byte-identical and the paired difference is exactly zero.
- `Fixed route w/ skip` — the biggest mover, +4.4 pp. It chooses which stops to
  skip from current demand, so a locked prefix forces it to visit stops whose
  demand has since been served or abandoned.
- `Window 900s` — +2.0 pp served, −35 s mean wait.

The windowed columns are the open question. `window-width-sweep-2.ipynb` found
coverage peaking exactly at the abandonment threshold, and that argument was
about passenger patience rather than commitment, so the peak should stay at
900 s. If it moves, the triage explanation needs revisiting.

In [ ]:
%matplotlib inline
# Static comparison charts only, so the inline backend is all we need.

import statistics
import time
from typing import Callable

from scipy import stats
from IPython.display import Markdown, display

from src.algorithm.DispatchPolicy import DispatchPolicy
from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import (
    ComparisonSummary,
    TrialResult,
    plot_kpi_bars,
    plot_wait_distribution,
    run_trial,
    summarize_comparison,
)
from src.map.PlanarMapProvider import PlanarMapProvider
from src.map.TransitGraph import TransitGraph
from src.map.graph import PlanarStreetGraph

# Trials per column. The original comparison ran 100; 30 keeps this notebook
# under a quarter of an hour while leaving the served-fraction interval around
# a point, and the paired differences much tighter than that.
N_TRIALS = 30
BASE_SEED = 0
COMMITMENTS = [0, 2]

# Swept service windows in seconds, matching window-insertion-comparison.
WINDOWS = [450.0, 600.0, 750.0, 900.0]

# Stands in for "no window at all": large enough that the deadline never
# binds, isolating what persistent insertion alone builds.
NO_WINDOW = 1e9

# run_trial kwargs matching every previous campus experiment. NOTE: max_wait_s
# here is the *backend's abandonment threshold*, a different quantity from a
# policy's service window.
LOAD = dict(max_requests=200, capacity=40, max_wait_s=900.0)


def window_factory(
    window: float,
) -> Callable[[TransitGraph], WindowInsertionPolicy]:
    """
    Build a WindowInsertionPolicy factory bound to one service window.

    A named closure rather than a lambda, so the late-binding trap that bites
    list comprehensions cannot arise.
    :param window: Service window in seconds.
    :return: Callable taking a TransitGraph and returning a policy.
    """
    def factory(graph: TransitGraph) -> WindowInsertionPolicy:
        return WindowInsertionPolicy(graph, max_wait_s=window)
    return factory


POLICIES: list[tuple[str, Callable[[TransitGraph], DispatchPolicy]]] = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Fixed route w/ skip", lambda g: SkipStopFixedRoutePolicy(g)),
    ("Insertion (no window)", window_factory(NO_WINDOW)),
] + [
    (f"Window {int(window)}s", window_factory(window))
    for window in WINDOWS
]

## 1. Shared map

Identical to every previous sweep — same `MAP_SEED`, same campus configuration
— so the `commitment=2` columns below are directly comparable to the committed
run in `window-insertion-comparison.ipynb`.

In [ ]:
MAP_SEED = 42

graph = PlanarStreetGraph.generate(
    n_seeds=17,
    bounds=(0.0, 0.0, 2000.0, 2000.0),
    seed=MAP_SEED,
    stop_density=0.5,
)

provider = PlanarMapProvider(graph)
print(f"{len(provider.get_stops())} bus stops on this map")

## 2. Cost check

Fourteen columns (seven policies at two commitment levels). One trial of the
most expensive policy is timed before committing to the rest — the fixed-route
columns are cheaper, so this over-estimates.

In [ ]:
start = time.perf_counter()
run_trial(
    policy_factory=window_factory(900.0),
    graph=graph,
    seed=BASE_SEED,
    silent=True,
    commitment=0,
    **LOAD,
)
probe_cost = time.perf_counter() - start

projected_s = N_TRIALS * len(POLICIES) * len(COMMITMENTS) * probe_cost
display(Markdown(
    f"- One trial, **Window 900s at commitment=0**: {probe_cost:.1f} s\n\n"
    f"Upper bound for the run below "
    f"({N_TRIALS} trials x {len(POLICIES)} policies x "
    f"{len(COMMITMENTS)} levels): **{projected_s / 60:.1f} min**"
))

## 3. The run

Labels are reused across the two calls, which is safe because each
`summarize_comparison` returns its own results mapping — nothing is shared
between them.

In [ ]:
summaries: dict[int, ComparisonSummary] = {}
for commitment in COMMITMENTS:
    summaries[commitment] = summarize_comparison(
        policies=POLICIES,
        graph=graph,
        n_trials=N_TRIALS,
        base_seed=BASE_SEED,
        commitment=commitment,
        **LOAD,
    )

## 4. The new baseline

This is the table that replaces the headline in
`window-insertion-comparison.ipynb`. Everything below `commitment=0`.

In [ ]:
display(Markdown("### `commitment=0` — the new baseline\n\n"
                 + summaries[0].markdown))
display(Markdown("### `commitment=2` — the old default, for reference\n\n"
                 + summaries[2].markdown))

## 5. What actually changed

Paired per-trial differences, `commitment=0` minus `commitment=2`. Positive is
better for served fraction and fulfilled count; negative is better for the
timing rows.

In [ ]:
def paired_delta(
    baseline: list[TrialResult],
    treatment: list[TrialResult],
    extract: Callable[[TrialResult], float | None],
) -> tuple[float, float, float]:
    """
    Mean per-trial difference (treatment - baseline) with a CI and a p-value.

    Trials are paired by index, which is meaningful because every column ran
    the same base_seed and trial count, so trial i faced the same demand
    sequence in both.
    :param baseline: Per-trial results for the reference column.
    :param treatment: Per-trial results for the column being tested.
    :param extract: Maps a TrialResult to the scalar being compared, or None
        when that trial does not define it.
    :return: (mean difference, 95% CI half-width, two-sided paired p-value).
        The last two are NaN with fewer than two usable pairs, and the p-value
        is NaN when every difference is identical (a policy the setting does
        not reach at all).
    """
    pairs = [
        (before, after)
        for base_trial, treat_trial in zip(baseline, treatment)
        if (before := extract(base_trial)) is not None
        and (after := extract(treat_trial)) is not None
    ]
    if not pairs:
        return float("nan"), float("nan"), float("nan")

    differences = [after - before for before, after in pairs]
    mean_difference = statistics.fmean(differences)
    if len(differences) < 2:
        return mean_difference, float("nan"), float("nan")

    half_width = (
        stats.t.ppf(0.975, len(differences) - 1)
        * statistics.stdev(differences)
        / len(differences) ** 0.5
    )
    p_value = float(stats.ttest_rel(
        [after for _, after in pairs],
        [before for before, _ in pairs],
    ).pvalue)
    return mean_difference, half_width, p_value


def served_pp(trial: TrialResult) -> float | None:
    """
    Served fraction as percentage points, or None when undefined.
    :param trial: One trial's results.
    :return: Served fraction x 100, or None.
    """
    if trial.kpi.served_fraction is None:
        return None
    return 100.0 * trial.kpi.served_fraction


# (display name, extractor, direction that counts as an improvement)
_MetricSpec = tuple[str, Callable[[TrialResult], float | None], str]

DELTA_METRICS: list[_MetricSpec] = [
    ("Served (pp)", served_pp, "higher"),
    ("Mean wait (s)", lambda r: r.kpi.mean_wait_time, "lower"),
    ("Mean ride (s)", lambda r: r.kpi.mean_ride_time, "lower"),
]

lines = [
    "| Policy | " + " | ".join(name for name, _, _ in DELTA_METRICS) + " |",
    "| --- | " + " | ".join("---" for _ in DELTA_METRICS) + " |",
]
for policy_name, _ in POLICIES:
    cells = []
    for _, extract, better in DELTA_METRICS:
        delta, half_width, p_value = paired_delta(
            summaries[2].results[policy_name],
            summaries[0].results[policy_name],
            extract,
        )
        improved = delta > 0 if better == "higher" else delta < 0
        mark = "**" if p_value < 0.05 and improved else ""
        cells.append(f"{mark}{delta:+.1f} ± {half_width:.1f}{mark} "
                     f"(p={p_value:.3f})")
    lines.append(f"| {policy_name} | " + " | ".join(cells) + " |")
display(Markdown("\n".join(lines)))
display(Markdown(
    "*Bold marks a difference both significant at p < 0.05 and in the "
    "favourable direction. `nan` in a p-value column means every paired "
    "difference was identical — for `Fixed route` that is the expected "
    "result, not a failure.*"
))

## 6. Charts

Both drawn from the `commitment=0` runs, i.e. the new baseline.

In [ ]:
plot_kpi_bars(
    summaries[0].results,
    fields=["served_fraction", "mean_wait_time",
            "mean_ride_time", "p90_wait_time"],
    title="New baseline at commitment=0",
)
plot_wait_distribution(summaries[0].results)

## What to look for

**The regression check first.** The `commitment=2` table should reproduce
`window-insertion-comparison.ipynb`: 72.6% served for skip-stop, 80.1% for the
no-window ablation, 91.6% for `Window 900s`. It ran 100 trials to this
notebook's 30, so expect wider intervals rather than different means. If the
means disagree, stop and resolve that before using anything else here.

**Then the new baseline.** These are the numbers that go into the docs and that
any future policy has to beat. Two things to read off:

- **Does `Window 900s` still win, and by how much?** Skip-stop is the column
  that gains most from the commitment change, so the margin should narrow. It
  narrowing is fine; it vanishing would be a finding.
- **Does the coverage peak stay at the 900 s window?** The triage argument in
  `window-width-sweep-2.ipynb` was about passenger patience, not commitment, so
  it should. A peak that has moved to 750 s or below means commitment was
  confounding that result and the sweep needs re-running at `commitment=0`.

**The delta table** is the provenance for the change. `Fixed route` reporting
exactly `+0.0 ± 0.0` with a `nan` p-value is the correct answer — identical
routes give zero-variance differences — and it doubles as a check that the
paired machinery is wired up correctly.

**Watch the ride rows.** Mean ride was the one metric where `commitment=0` was
not clearly better at 200 requests in the confirmation run (p = 0.25, though it
reached significance at 500). If a windowed column shows a significant ride
*regression* here, that matters: ride tail is exactly where the planned
reordering work is expected to pay off, and a confounded baseline would make
those results unreadable.

### A caveat on reproducibility

Set `PYTHONHASHSEED=0` before launching Jupyter. `FixedRoutePolicy` generates
its cycle by breaking ties with `min()` over a `set[str]`, so without it the
fixed-route columns move between processes.

### After this runs

The new `commitment=0` numbers should replace the tuning table in
`autobus-docs/docs/concepts/window_insertion.md`, which is currently flagged as
having been measured at the old default.